# Extracting ECG data from DICOM files
The ecgprocess.py module wraps pydicom to standardise and extract patient information tables, as well as lead-specific waveforms and median beats.

In [1]:
import os
import tempfile
import numpy as np
import pandas as pd
import ecgprocess.process_dicoms as pdicom
from ecgprocess.process_dicoms import (
    ECGDICOMReader,
    ECGDICOMTable,

)
from ecgprocess.example_data.examples import (
    list_data_paths,
)

# Example file
dicom_file_path = list_data_paths()['example_dicom_1']


## The ECGDICOMReader dataclass 

In [2]:
ECG_DCM = ECGDICOMReader()
dicom_record = ECG_DCM(path=dicom_file_path)

### Participant information

In [3]:
# patient information
pd.Series(dicom_record.GeneralInfo)

SOPinstanceUID                2.25.269796857626990821315969488216511468638
SERIESinstanceUID             2.25.242602932404839926316317069310692902961
STUDYinstanceUID               2.25.70398906267875282906347616363167587031
PatientID                                                          removed
PatientBirthDate                                                          
PatientName                  (1, 2, -, l, e, a, d, -, e, x, a, m, p, l, e)
PatientSex                                                               F
StudyDescription                                           AMC 12 LEAD ECG
AcquisitionTimeZone                                                  +0100
AccessionNumber                                                           
SamplingFrequency                                                      500
SamplingFrequencyOriginal                                            500.0
ChannelNumber                                                           12
ChannelSensitivity       

### Waveforms and Median beats

In [4]:
dicom_record.Waveforms

{'I': array([  0.  ,   0.  ,  29.28, ..., -48.8 , -53.68, -34.16]),
 'II': array([   0.  ,   -9.76,    9.76, ..., -102.48,  -82.96,  -58.56]),
 'III': array([  0.  ,  -9.76, -19.52, ..., -53.68, -29.28, -24.4 ]),
 'aVR': array([  0.  ,   4.88, -19.52, ...,  73.2 ,  68.32,  43.92]),
 'aVL': array([ 0.  ,  4.88, 24.4 , ...,  0.  , -9.76, -4.88]),
 'aVF': array([  0.  ,  -9.76,  -4.88, ..., -78.08, -53.68, -39.04]),
 'V1': array([-4.88,  0.  , -9.76, ..., 24.4 , 24.4 , 24.4 ]),
 'V2': array([ 9.76, 24.4 , 24.4 , ...,  0.  ,  0.  ,  0.  ]),
 'V3': array([24.4 , 34.16, 29.28, ...,  0.  ,  0.  ,  0.  ]),
 'V4': array([ 29.28,  34.16,  39.04, ..., -29.28, -34.16, -29.28]),
 'V5': array([-14.64, -14.64,  -9.76, ..., -78.08, -82.96, -78.08]),
 'V6': array([-34.16, -29.28, -24.4 , ..., -87.84, -92.72, -87.84])}

In [5]:
dicom_record.MedianWaveforms['I'][0:20]

array([ -4.88,  -4.88,  -4.88,  -9.76,  -9.76,  -9.76,  -9.76,  -4.88,
        -4.88,  -4.88,  -4.88,  -4.88,  -9.76,  -9.76,  -9.76, -14.64,
       -14.64,  -4.88,  -4.88,  -4.88])

## Custom behaviour 
The class object has a number of attributes (dictionaries) which can be used to extract more (or less) data from pydicom instance. 

In [6]:
# extract any comments 

# I add the comments name to the METADATA dict
# here the value should map to the pydicom attribute name 
# and the key to the new name we want to use internally
ECG_DCM.METADATA['Image Comment'] = 'ImageComments'

# II add the name to the RESULTS_DICT
ECG_DCM.RESULTS_LIST = ECG_DCM.RESULTS_LIST + ['Image Comment'] 
dicom_record_new = ECG_DCM(path=dicom_file_path)
dicom_record_new.GeneralInfo['Image Comment'] 

'For research purposes only'

## ECGDICOMTable 
A class to extract data from multiple records and map these to tabular data.

We will first simply initilise the class and differentiate between valid and invalid files. 

In [7]:

dicom_path_list = [
    list_data_paths()['example_dicom_1'],
    list_data_paths()['example_dicom_2'],
]
ecgreader = ECGDICOMReader()
ecgtable = ECGDICOMTable(ecgreader, path_list=dicom_path_list)
res = ecgtable(verbose=True)
print('Failed paths: {0}\nCurated paths: {1}.'.format(
    None if len(res.FailedPathList) == 0 else res.FailedPathList, 
    res.CuratedPathList))

Failed paths: None
Curated paths: ['/home/amand/google_drive/Research/ECGProcess/ecgprocess/example_data/example_datasets/example-DICOM1.dcm', '/home/amand/google_drive/Research/ECGProcess/ecgprocess/example_data/example_datasets/example-DICOM2.dcm'].


### Process the two files
The first try at this will raise an error because the headers do not match. 

In [8]:
try:
    res.get_table()
except Exception as e:
    print(f"\n\nAn error occurred: {e}")

Processing path: /home/amand/google_drive/Research/ECGProcess/ecgprocess/example_data/example_datasets/example-DICOM1.dcm.
Processing path: /home/amand/google_drive/Research/ECGProcess/ecgprocess/example_data/example_datasets/example-DICOM2.dcm.


An error occurred: "The dictionaries contain distinct keys. The last set of valid keys was ['I', 'II', 'III', 'aVR', 'aVL', 'aVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6'], compared to ['I (Einthoven)', 'II', 'III', 'aVR', 'aVL', 'aVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']."


Next will solve this by providing a dictionary mapping the old name to the new name. These dictionaries can be expanded for multiple potential names and ignored if there are now matches. This will results in a long table with multiple files row-binded.

In [9]:
combined=res.get_table(update_keys={'I (Einthoven)': 'I'})
combined.WaveFormsTable

Processing path: /home/amand/google_drive/Research/ECGProcess/ecgprocess/example_data/example_datasets/example-DICOM1.dcm.
Processing path: /home/amand/google_drive/Research/ECGProcess/ecgprocess/example_data/example_datasets/example-DICOM2.dcm.


,SOPinstanceUID,SamplingSequence,Lead,Voltage,Waveform type
0,2.25.269796857626990821315969488216511468638,0,I,0.00,rhythm
1,2.25.269796857626990821315969488216511468638,1,I,0.00,rhythm
2,2.25.269796857626990821315969488216511468638,2,I,29.28,rhythm
3,2.25.269796857626990821315969488216511468638,3,I,43.92,rhythm
4,2.25.269796857626990821315969488216511468638,4,I,48.80,rhythm
...,...,...,...,...,...
119995,1.3.6.1.4.1.40744.65.2218354498692802781165553...,4995,V6,-195.00,rhythm
119996,1.3.6.1.4.1.40744.65.2218354498692802781165553...,4996,V6,-190.00,rhythm
119997,1.3.6.1.4.1.40744.65.2218354498692802781165553...,4997,V6,-190.00,rhythm
119998,1.3.6.1.4.1.40744.65.2218354498692802781165553...,4998,V6,-185.00,rhythm


### Write to disk
These tables can of course be written to disk. If one simply wants to combined the individual files and write these to disk a more memory efficient solution is to extract each individual DICOM and append them to a single file. This is what `write_ecg` does ... Actually it writes four files to disk!

In [10]:
with tempfile.TemporaryDirectory() as temp_dir:
    combined=res.write_ecg(update_keys={'I (Einthoven)': 'I'}, target_path=temp_dir)
    # list saved files
    print('Directory content:')
    for file_name in os.listdir(getattr(combined, 'target_path')):
        print('   - {}'.format(file_name))

Processing path: /home/amand/google_drive/Research/ECGProcess/ecgprocess/example_data/example_datasets/example-DICOM1.dcm.
Processing path: /home/amand/google_drive/Research/ECGProcess/ecgprocess/example_data/example_datasets/example-DICOM2.dcm.
Directory content:
   - GeneralInfoTable.tsv.gz
   - WaveFormsTable.tsv.gz
   - FailedFiles.txt
   - MedianWaveTable.tsv.gz


In [11]:
help(pdicom)

Help on module ecgprocess.process_dicoms in ecgprocess:

NAME
    ecgprocess.process_dicoms

DESCRIPTION
    A module to extract median beats and raw waveforms from DICOM ECGs wrapping
    pydicom.

CLASSES
    builtins.object
        ECGDICOMReader
        ECGDICOMTable

    class ECGDICOMReader(builtins.object)
     |  ECGDICOMReader(augment_leads: bool = False, resample_500: bool = True) -> None
     |
     |  Takes an ECG DICOM file and extracts metadata, median beats (if available)
     |  and raw waveforms.
     |
     |  Parameters
     |  ----------
     |  augment_leads: bool, default `False`
     |      Whether the augmented leads are available in the DICOM, if not
     |      these are calculated.
     |  resample: bool, default `True`
     |      Whether to resample the ECG to a frequency of 500 Hertz.
     |
     |  Attributes
     |  ----------
     |  augment_leads: bool, default `False`
     |      Whether the augmented leads are available in the DICOM, if not
     |   